- Load dataset/malicious_phish.csv.

- Check the dataset columns and labels.

- Clean missing and duplicate URLs.

- Create binary classification labels: 0 = Benign, 1 = Malicious.

- Save the processed CSV files.

- Print one final summary.

In [1]:
import pandas as pd
import numpy as np
import re
import json
import random

from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# Works when notebook is inside PhishingDetection/notebooks
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH = PROJECT_ROOT / "dataset" / "malicious_phish.csv"
OUTPUT_DIR = PROJECT_ROOT / "dataset" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"

In [2]:
df = pd.read_csv(DATASET_PATH, low_memory=False)

original_rows = len(df)

required_columns = {"url", "type"}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {df.columns.tolist()}"
    )

original_class_counts = (
    df["type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .value_counts()
    .to_dict()
)

In [3]:
df = df[["url", "type"]].copy()

df = df.dropna(subset=["url", "type"])

df["url"] = df["url"].astype(str).str.strip()

df["type"] = (
    df["type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_classes = [
    "benign",
    "phishing",
    "malware",
    "defacement"
]

df = df[df["type"].isin(valid_classes)].copy()

df = df[df["url"].str.len() > 0].copy()

# Check conflicting labels before removing duplicates
conflicting_urls = (
    df.groupby("url")["type"]
    .nunique()
)

conflicting_urls = conflicting_urls[
    conflicting_urls > 1
].index

conflicting_count = len(conflicting_urls)

# Exclude URLs with conflicting original labels
df = df[~df["url"].isin(conflicting_urls)].copy()

df = df.drop_duplicates(subset=["url"])

df = df.reset_index(drop=True)

In [4]:
binary_mapping = {
    "benign": 0,
    "phishing": 1,
    "malware": 1,
    "defacement": 1
}

multiclass_mapping = {
    "benign": 0,
    "defacement": 1,
    "phishing": 2,
    "malware": 3
}

df["label_binary"] = df["type"].map(binary_mapping).astype(int)

df["label_multi"] = df["type"].map(multiclass_mapping).astype(int)

# Preserve original URL text for all models
df["clean_url"] = df["url"]

df = df[
    [
        "url",
        "type",
        "label_binary",
        "label_multi",
        "clean_url"
    ]
]

In [5]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["type"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["type"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Verify no exact URL overlaps
train_urls = set(train_df["url"])
val_urls = set(val_df["url"])
test_urls = set(test_df["url"])

assert train_urls.isdisjoint(val_urls)
assert train_urls.isdisjoint(test_urls)
assert val_urls.isdisjoint(test_urls)

In [6]:
train_df.to_csv(
    OUTPUT_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    OUTPUT_DIR / "val.csv",
    index=False
)

test_df.to_csv(
    OUTPUT_DIR / "test.csv",
    index=False
)

metadata = {
    "random_seed": SEED,
    "task": "binary_malicious_url_detection",
    "label_mapping": binary_mapping,
    "multiclass_mapping": multiclass_mapping,
    "original_rows": original_rows,
    "cleaned_rows": len(df),
    "conflicting_urls_removed": conflicting_count,
    "train_rows": len(train_df),
    "validation_rows": len(val_df),
    "test_rows": len(test_df),
    "split_ratio": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15
    }
}

with open(
    OUTPUT_DIR / "metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=4)

In [7]:
print("=" * 60)
print("DATA PREPARATION COMPLETED")
print("=" * 60)

print(f"Original dataset:       {original_rows:,}")
print(f"Cleaned dataset:        {len(df):,}")
print(f"Conflicting URLs:       {conflicting_count:,}")

print("-" * 60)

print(f"Training samples:       {len(train_df):,}")
print(f"Validation samples:     {len(val_df):,}")
print(f"Testing samples:        {len(test_df):,}")

print("-" * 60)

print("Binary class distribution:")

for label, count in sorted(
    df["label_binary"].value_counts().items()
):
    name = "Benign" if label == 0 else "Malicious"
    print(f"{name:<20} {count:,}")

print("-" * 60)

print("Original four-class distribution:")

for name, count in df["type"].value_counts().items():
    print(f"{name:<20} {count:,}")

print("-" * 60)

print(f"Saved datasets to: {OUTPUT_DIR}")
print("Train/validation/test overlap: NONE")
print("=" * 60)

DATA PREPARATION COMPLETED
Original dataset:       651,191
Cleaned dataset:        641,113
Conflicting URLs:       6
------------------------------------------------------------
Training samples:       448,779
Validation samples:     96,167
Testing samples:        96,167
------------------------------------------------------------
Binary class distribution:
Benign               428,074
Malicious            213,039
------------------------------------------------------------
Original four-class distribution:
benign               428,074
defacement           95,308
phishing             94,086
malware              23,645
------------------------------------------------------------
Saved datasets to: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\dataset\processed
Train/validation/test overlap: NONE
